
# Variational Autoencoder (VAE) – Workshop Notebook

This workshop notebook is designed for **hands-on understanding** of Variational Autoencoders.

Target audience:
- Beginner to intermediate deep learning learners
- Students transitioning from Autoencoders to Probabilistic Models

You will learn:
- Why VAE is probabilistic
- Encoder–Decoder with latent distribution
- Reparameterization trick
- KL Divergence in practice


## 1. Imports and Setup

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt


## 2. Hyperparameters

In [ ]:

batch_size = 128
epochs = 20
lr = 1e-3

latent_dim = 20
image_dim = 28 * 28


## 3. Dataset
MNIST is ideal for visualizing latent space behavior.

In [ ]:

transform = transforms.Compose([
    transforms.ToTensor()
])

dataset = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


## 4. VAE Architecture
Encoder outputs mean and log-variance.

In [ ]:

class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(image_dim, 400),
            nn.ReLU()
        )
        self.mu = nn.Linear(400, latent_dim)
        self.logvar = nn.Linear(400, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 400),
            nn.ReLU(),
            nn.Linear(400, image_dim),
            nn.Sigmoid()
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.mu(h), self.logvar(h)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar


## 5. Loss Function
Reconstruction loss + KL Divergence

In [ ]:

def vae_loss(recon_x, x, mu, logvar):
    recon_loss = nn.functional.binary_cross_entropy(
        recon_x, x, reduction='sum'
    )
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl


## 6. Training Loop

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)

for epoch in range(epochs):
    total_loss = 0
    for data, _ in loader:
        data = data.view(-1, image_dim).to(device)

        recon, mu, logvar = model(data)
        loss = vae_loss(recon, data, mu, logvar)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(dataset):.2f}")


## 7. Sampling from Latent Space

In [ ]:

with torch.no_grad():
    z = torch.randn(16, latent_dim).to(device)
    samples = model.decoder(z).view(-1, 1, 28, 28)

plt.imshow(samples[0][0], cmap="gray")
plt.title("Generated Sample")
plt.axis("off")


## 8. Key Takeaways
- VAE learns smooth latent space
- Sampling is mathematically grounded
- KL Divergence regularizes encoding